# Recomendaciones en Netflix usando una RBM

In [1]:
import pandas as pd
import numpy as np
from sklearn.neural_network import BernoulliRBM
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

c:\Users\luis2\proyectos\Movie_Recommendation\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/netflix_titles.csv")

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [4]:
print("Número de características:", len(df.columns))
print("Longitud del conjunto de datos:", len(df))

Número de características: 12
Longitud del conjunto de datos: 8807


In [5]:
# Visualizamos los tipos de cada uno de los atributos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [6]:
# Comprobamos si alguna columna tiene valores nulos
df.isna().any()

show_id         False
type            False
title           False
director         True
cast             True
country          True
date_added       True
release_year    False
rating           True
duration         True
listed_in       False
description     False
dtype: bool

In [7]:
# Numero de nulos en cada columna
df.isna().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [8]:
# Se rellenan los valores nulos con el valor "Unknown"
df = df.fillna("Unknown")

In [9]:
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Unknown,Unknown,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


Se utiliza un modelo de huggingface "all-MiniLM-L6-v2" para transformar la descripción de las películas a embeddings

#### Liga al modelo
https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

#### Descripción del modelo
This is a sentence-transformers model: It maps sentences & paragraphs to a 384 dimensional dense vector space and can be used for tasks like clustering or semantic search.

In [10]:
# transformamos el texto a embeddings usando un modelo de huggingface
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

desc_embeddings = model.encode(df['description'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 276/276 [00:36<00:00,  7.53it/s]


In [11]:
description_processed = {
    "desc_embeddings": desc_embeddings
}

In [12]:
# Guardado
np.savez("../data/netflix_processed.npz", **description_processed)

In [13]:
# Carga de datos de descripción
data = np.load("../data/netflix_processed.npz", allow_pickle=True)
desc_embeddings = data["desc_embeddings"]

In [14]:
# El dataset no cuenta con usuarios, por lo que los simulamos con una distribución bionomial
np.random.seed(42)

n_users = 1000
n_items = len(df)

# Probabilidad base de ver una película
interaction_prob = 0.08

user_item_matrix = np.random.binomial(
    1,
    interaction_prob,
    size=(n_users, n_items)
)

user_item_matrix.shape

(1000, 8807)

In [15]:
# Primer elemento de la matiz de usuarios (1-vio la pelicula, 0-no vio la pelicula)
user_item_matrix[0]

array([0, 1, 0, ..., 0, 0, 0], shape=(8807,), dtype=int32)

### Entrenamiento de una RBM para recomendar peliculas en Netflix

In [16]:
# Entrenamiento RBM

rbm = BernoulliRBM(
    n_components=128,
    learning_rate=0.01,
    batch_size=64,
    n_iter=25,
    verbose=True,
    random_state=42
)

rbm.fit(user_item_matrix)

[BernoulliRBM] Iteration 1, pseudo-likelihood = -5723.00, time = 0.95s
[BernoulliRBM] Iteration 2, pseudo-likelihood = -5496.84, time = 1.04s
[BernoulliRBM] Iteration 3, pseudo-likelihood = -5289.43, time = 1.05s
[BernoulliRBM] Iteration 4, pseudo-likelihood = -5100.97, time = 1.05s
[BernoulliRBM] Iteration 5, pseudo-likelihood = -4932.25, time = 1.07s
[BernoulliRBM] Iteration 6, pseudo-likelihood = -4777.90, time = 1.19s
[BernoulliRBM] Iteration 7, pseudo-likelihood = -4635.63, time = 1.11s
[BernoulliRBM] Iteration 8, pseudo-likelihood = -4501.94, time = 1.14s
[BernoulliRBM] Iteration 9, pseudo-likelihood = -4386.31, time = 1.08s
[BernoulliRBM] Iteration 10, pseudo-likelihood = -4269.22, time = 1.06s
[BernoulliRBM] Iteration 11, pseudo-likelihood = -4159.98, time = 1.04s
[BernoulliRBM] Iteration 12, pseudo-likelihood = -4066.61, time = 1.08s
[BernoulliRBM] Iteration 13, pseudo-likelihood = -3973.72, time = 1.04s
[BernoulliRBM] Iteration 14, pseudo-likelihood = -3881.90, time = 1.05s
[

,"n_components n_components: int, default=256Number of binary hidden units.",128
,"learning_rate learning_rate: float, default=0.1The learning rate for weight updates. It is *highly* recommendedto tune this hyper-parameter. Reasonable values are in the10**[0., -3.] range.",0.01
,"batch_size batch_size: int, default=10Number of examples per minibatch.",64
,"n_iter n_iter: int, default=10Number of iterations/sweeps over the training dataset to performduring training.",25
,"verbose verbose: int, default=0The verbosity level. The default, zero, means silent mode. Rangeof values is [0, inf].",True
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for:- Gibbs sampling from visible and hidden layers.- Initializing components, sampling from layers during fit.- Corrupting the data when scoring samples.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42


In [17]:
# Función que hace las recomendaciones usando la RBM

def recommendation_rbm(rbm, user_vector, top_k=10):

    # la rbm reconstruye el vector del usuario
    reconstructed = rbm.gibbs(user_vector.reshape(1, -1))

    scores = reconstructed.flatten()
    
    # evitar recomendar vistos, en vistos scores = False
    scores[user_vector == 1] = False

    # selecciona las peliculas recomendadas
    top_items = np.argsort(scores)[-top_k:][::-1]
    
    return top_items, scores

In [18]:
# Función que rankea el contenido a recomendar

def rank_content(user_vector, candidate_items, embeddings):

    # se detectan las peliculas que ya vio el usuario
    seen_items = np.where(user_vector == 1)[False]

    # se toman los embeddings de las peliculas ya vistas y se obtiene un vector de preferencias
    user_profile = embeddings[seen_items].mean(axis=0)

    # por medio del producto punto se puede saber que tanto se parece una pelicula a otra
    # K(X, Y) = <X, Y> / (||X||*||Y||)
    sims = cosine_similarity(
        [user_profile],
        embeddings[candidate_items]
    )[0]

    # se ordenan las peliculas mas relevantes
    ranked = [x for _, x in sorted(zip(sims, candidate_items), reverse=True)]
    
    return ranked

In [ ]:
# Funcionamiento del sistema de recomendación

def recommendation_system(user_id, top_k=10):
    
    user_vector = user_item_matrix[user_id]
    
    # RBM
    candidates, scores = recommendation_rbm(rbm, user_vector, top_k=10)
    
    # Ranking con embeddings
    final_recommendations = rank_content(
        scores,
        candidates,
        desc_embeddings
    )
    
    return final_recommendations[:top_k]

In [20]:
# Test: Recomendaciones para el usuario id=0
user_id = 0

recommendations = recommendation_system(user_id, top_k=10)

df.iloc[recommendations][["title", "listed_in", "release_year"]]

,title,listed_in,release_year
4711,On Children,"International TV Shows, TV Dramas, TV Mysteries",2018
4756,Iliza Shlesinger: Elder Millennial,Stand-Up Comedy,2018
4914,海的儿子,"International TV Shows, TV Dramas",2016
4796,My Birthday Song,"Dramas, International Movies, Thrillers",2018
4901,The Clapper,"Comedies, Dramas, Independent Movies",2017
4734,Tamasha,"Dramas, International Movies, Romantic Movies",2015
4866,Carlos Ballarta: Furia Ñera,Stand-Up Comedy,2018
4963,The Titan,"Dramas, Sci-Fi & Fantasy, Thrillers",2018
4937,"Ram Dass, Going Home","Documentaries, Faith & Spirituality",2018
4762,Home: Adventures with Tip & Oh,"Kids' TV, TV Comedies",2018


In [21]:
import joblib

# Guardar modelo RBM

joblib.dump(rbm, "../models/rbm_model.pkl")

# Guardar matriz usuario-item

np.save("../data/user_item_matrix.npy", user_item_matrix)